Analyzed conv-layer.ipynb:L1-500
Analyzed conv-layer.ipynb:L500-556

本笔记本 [conv-layer.ipynb](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/pytorch/chapter_convolutional-neural-networks/conv-layer.ipynb) 详细讲解了**卷积层（Convolutional Layer）**的底层数学本质、代码实现以及其在图像特征提取中的应用与参数自学习机制。

---

### 一、 核心内容概览

#### 1. 互相关运算（Cross-Correlation）
* **运算机制**：在深度学习中，所谓的“卷积层”实际上执行的是**二维互相关运算**。卷积窗口在输入张量上从左到右、从上到下滑动，与核张量做元素对应相乘并求和，生成对应位置的标量输出。
* **输出尺寸公式**：
  若输入大小为 `n_h × n_w`，卷积核大小为 `k_h × k_w`（无填充、步长为 1）：
  ```text
  输出形状 = (n_h - k_h + 1) × (n_w - k_w + 1)
  ```
* **基础实现**： Notebook 中从零实现了二维互相关运算函数 `corr2d(X, K)`。

---

#### 2. 二维卷积层结构
* **可学习参数**：
  * **卷积核权重（`weight`）**：形状为 `(k_h, k_w)` 的张量。
  * **标量偏置（`bias`）**：通常初始化为 0。
* **前向传播公式**：
  ```text
  Y = corr2d(X, W) + b
  ```
* **面向对象封装**：继承 `nn.Module` 自定义封装了基础的 `Conv2D` 层。

---

#### 3. 经典案例：边缘检测
* **垂直边缘检测核**：设计卷积核 `K = [[1.0, -1.0]]`。
* **物理意义**：
  * 水平方向相邻两像素相同（平坦区域）时，输出为 `0`；
  * 从白色（`1`）过渡到黑色（`0`）时，输出为 `1`；
  * 从黑色（`0`）过渡到白色（`1`）时，输出为 `-1`。
* **局限性**：该核只能检测垂直边缘；输入转置后（水平边缘），其输出全为 `0`。

---

#### 4. 卷积核参数的自动学习
* **问题背景**：复杂特征核无法人工手写，必须通过数据驱动学习。
* **训练流程**：
  * 使用输入图像 `X` 与目标特征图 `Y`；
  * 构建 `nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)` 并随机初始化；
  * 损失函数采用均方误差平方和 `loss = ∑ (Y_hat - Y)²`；
  * 经过 10 轮梯度下降更新后，网络自学习出的权重收敛至非常接近人工设计的 `[[1.0, -1.0]]`。

---

#### 5. 互相关 vs 严格卷积
* **数学差异**：严格数学定义的二维卷积需要先将卷积核进行**水平与垂直双向翻转**（即旋转 180°），然后再与输入做互相关。
* **深度学习中的等价性**：由于卷积核本身是随机初始化并通过数据反向传播学习出来的，翻转与否仅改变核权重的空间排列，并不影响模型的特征提取能力。因此在深度学习文献中，通常直接将“互相关运算”称作“卷积运算”。

---

#### 6. 特征映射（Feature Map）与感受野（Receptive Field）
* **特征映射**：卷积层输出的二维张量，代表输入经过空间特征提取后的表征。
* **感受野**：指前向传播过程中，输入层（或上一层）中能够影响当前特征图某特定输出元素的区域范围。
* **核心启示**：网络层数越深，深层神经元的**感受野越大**，从而能够捕获更大尺度、更全局的语义信息。

---

### 二、 核心代码结构速览

```python
# 1. 二维互相关运算核心实现
def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

# 2. 基于 PyTorch 的自学习卷积核循环
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
```

---

### 三、 本章核心要点总结
1. **本质是互相关**：深度学习中的卷积层底层计算绝大部分是互相关运算。
2. **局部性与平移不变性**：卷积核滑动窗口共享权重，使得参数量远小于全连接层。
3. **特征由浅入深**：通过卷积核学习不同方向/尺度的边缘、纹理等模式，叠加多层可扩大感受野以抽取抽象语义。

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
def corr2d(X, K):
    h,w = K.shape
    Y = torch.zeros(size=(X.shape[0]-h+1, X.shape[1]-w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j]=(X[i:i+h, j:j+w] * K).sum()
    return Y


In [7]:
X = torch.tensor(range(9)).reshape((3,3))

In [8]:
X

tensor([[0, 1, 2],
        [3, 4, 5],
        [6, 7, 8]])

In [11]:
K = torch.tensor(range(4)).reshape((2,2))

In [12]:
corr2d(X,K)

tensor([[19., 25.],
        [37., 43.]])

In [14]:
class Conv2D(nn.Module):

    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))
    
    def forward(self, x):
        return corr2d(x, self.weight) + self.bias



In [15]:
X = torch.ones((6,8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [19]:
K = torch.tensor([[1,-1]])

In [20]:
K

tensor([[ 1, -1]])

In [21]:
Y = corr2d(X,K)

In [22]:
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [23]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [24]:
conv2d = nn.Conv2d(1,1, kernel_size=(1,2), bias=False)

X = X.reshape(1,1,6,8)
Y = Y.reshape(1,1,6,7)

lr = 3e-2
for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat-Y)**2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= lr*conv2d.weight.grad
    if (i+1)%2 ==0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 5.006
epoch 4, loss 0.840
epoch 6, loss 0.141
epoch 8, loss 0.024
epoch 10, loss 0.004


In [25]:
conv2d.weight.data.reshape(((1,2)))

tensor([[ 0.9879, -0.9888]])

Ran check Conv2D autograd · Exited with code 1
Canceled find python environment
Killed task Find python environment
Ran find python with pytorch · Exited with code 1
Ran find venv in workspace
Ran test Conv2D autograd
Ran run Conv2D training loop
Ran search exercise in workspace
Checked task Search exercise in workspace
Used tool: manage_task
Ran test Conv2D with 4D tensor
Ran test backward on 4D
Ran test Conv2D with 4D autograd
Ran train custom Conv2D on 2D
Ran verify diagonal edge experiment

以下是针对 [conv-layer.ipynb](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/pytorch/chapter_convolutional-neural-networks/conv-layer.ipynb) 课后练习题的逐题详尽解答：

---

### 练习 1：构建具有对角线边缘的图像 `X`

#### 问题描述
构建一个具有对角线边缘的图像 `X`。
1. 如果将本节举例的卷积核 `K` 应用于 `X`，会发生什么情况？
2. 如果转置 `X` 会发生什么？
3. 如果转置 `K` 会发生什么？

#### 解答与分析
我们构建一个下三角为 1、上三角为 0 的 `6 × 6` 图像作为具有主对角线边缘的输入：

```python
import torch

def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

# 构造具有对角线阶梯边缘的图像 X (下三角为1，其余为0)
X = torch.tril(torch.ones(6, 6))
# 原卷积核：检测水平方向跳变（形状为 1 × 2）
K = torch.tensor([[1.0, -1.0]])
```

图像 `X` 的形态如下：
```text
[[1., 0., 0., 0., 0., 0.],
 [1., 1., 0., 0., 0., 0.],
 [1., 1., 1., 0., 0., 0.],
 [1., 1., 1., 1., 0., 0.],
 [1., 1., 1., 1., 1., 0.],
 [1., 1., 1., 1., 1., 1.]]
```

#### 1. 将 `K` 应用于 `X`：
* **计算**：`Y = corr2d(X, K)`
* **现象**：
  ```text
  tensor([[1., 0., 0., 0., 0.],
          [0., 1., 0., 0., 0.],
          [0., 0., 1., 0., 0.],
          [0., 0., 0., 1., 0.],
          [0., 0., 0., 0., 1.],
          [0., 0., 0., 0., 0.]])
  ```
* **解释**：对角线边缘同时存在**水平方向**和**垂直方向**的像素突变。因为 `K = [[1.0, -1.0]]` 计算水平相邻两元素的差值（`x[i, j] - x[i, j+1]`），在对角线交界处由 1 跨到 0，产生差值为 `1.0`。因此在输出的主对角线位置检测出一条斜向的边缘线。

#### 2. 如果转置 `X`（即 `X.t()`）：
* **计算**：`corr2d(X.t(), K)`
* **现象**：
  ```text
  tensor([[ 0.,  0.,  0.,  0.,  0.],
          [-1.,  0.,  0.,  0.,  0.],
          [ 0., -1.,  0.,  0.,  0.],
          [ 0.,  0., -1.,  0.,  0.],
          [ 0.,  0.,  0., -1.,  0.],
          [ 0.,  0.,  0.,  0., -1.]])
  ```
* **解释**：转置后图像由下三角变为上三角矩阵，对角线交界的水平过渡方向由原本的“1 变 0”反转为“0 变 1”（`0 - 1 = -1.0`），因此检测到的对角线响应值全变为 `-1.0`，且位置相应向左下平移了一格。

#### 3. 如果转置 `K`（即 `K.t()`，形状变为 `2 × 1`）：
* **计算**：`corr2d(X, K.t())`
* **现象**：
  ```text
  tensor([[ 0., -1.,  0.,  0.,  0.,  0.],
          [ 0.,  0., -1.,  0.,  0.,  0.],
          [ 0.,  0.,  0., -1.,  0.,  0.],
          [ 0.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  0.,  0.,  0.,  0., -1.]])
  ```
* **解释**：`K.t()` 变为了垂直方向差分核（`[1.0], [-1.0]`），检测的是垂直方向上的像素跳变。在原图 `X` 中，从上到下看对角线边缘是由 0 变 1（`0 - 1 = -1.0`），因此同样能在对角线附近检测到斜向边缘，且响应值为 `-1.0`。

---

### 练习 2：在我们创建的 `Conv2D` 自动求导时，有什么错误消息？

#### 解答与分析
在书中自定义的简易类如下：
```python
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias
```

若直接将该自定义层代入后文代码进行自动求导训练，会遇到以下两类典型问题：

1. **输入张量维度不匹配（广播/形状错误）**：
   * 后文在使用内置 `nn.Conv2d` 时，输入被改为了 4D 张量 `(批量大小, 通道数, 高, 宽)`：
     ```python
     X = X.reshape((1, 1, 6, 8))
     Y = Y.reshape((1, 1, 6, 7))
     ```
   * 若直接输入给自定义的 `Conv2D`，`corr2d` 中的切片操作会将 `X.shape[0]`（值为 1）当做图像高度，导致计算得到的输出高度宽度为 `(1, 0)` 的空张量。在执行 `(Y_hat - Y) ** 2` 计算损失时，会抛出 RuntimeError 错误信息：
     ```text
     RuntimeError: The size of tensor a (0) must match the size of tensor b (7) at non-singleton dimension 3
     ```
2. **偏置标量梯度爆炸导致损失发散**：
   * 即使输入保持 2D 张量，`self.bias` 是一个标量，它在 `forward` 中直接与整个 `corr2d(x, self.weight)` 相加。
   * 特征图上的所有像素点（本例中 42 个元素）都对该标量偏置求导累加。未归一化时其梯度是单个像素梯度的 42 倍。直接使用 `lr = 0.03` 会导致 `bias` 瞬间梯度爆炸，数轮迭代后 `loss` 飙升至天文数字。这正是为何后文中直接使用 `bias=False` 的原因。
3. **框架历史背景（MXNet / 早期实现）**：
   * 在早期框架（如 MXNet Gluon）中，`corr2d` 里 `Y[i, j] = ...` 这种通过 Python 双重循环进行**原位赋值（in-place modification）**的操作无法被自动微分引擎追踪，会直接报错 `Cannot find gradient function for in-place assignment`。PyTorch 现代版本的 Autograd 引擎虽然通过动态计算图跟踪切片赋值，但双层 Python 循环效率极低且缺乏批处理与多通道支持。

---

### 练习 3：如何通过改变输入张量和卷积核张量，将互相关运算表示为矩阵乘法？

#### 解答与分析
这是现代深度学习框架（如 cuDNN）实现卷积加速的核心技术之一，主要有两种等价表示方法：

#### 方案一：`im2col` + GEMM（通用矩阵乘法，工业界标准做法）
1. **展开输入（`im2col` / `unfold`）**：
   * 将输入图像中每个滑动窗口所覆盖的局部小补丁（大小为 `k_h × k_w`）拉平成一个长度为 `k_h · k_w` 的行向量。
   * 共有 `N = (n_h - k_h + 1) · (n_w - k_w + 1)` 个滑动窗口，将这些行向量拼接为一个矩阵 `X_col`，形状为：
     ```text
     X_col:  N × (k_h · k_w)
     ```
2. **重塑卷积核**：
   * 将卷积核展平成一列：
     ```text
     K_col:  (k_h · k_w) × 1
     ```
   * （多通道时为 `(c_in · k_h · k_w) × c_out`）。
3. **一次矩阵乘法完成卷积**：
   ```text
   Y_col = X_col · K_col      # 结果形状为 N × 1
   ```
4. **Reshape 回空间尺寸**：将 `Y_col` 重新排列为 `(n_h - k_h + 1, n_w - k_w + 1)`。

#### 方案二：块 Toeplitz（托普利茨）矩阵法
1. 将输入图像展开为一维长向量 `x_vec`（长度为 `n_h · n_w`）。
2. 将卷积核展开并填充构造成一个大型的双重块循环/块 Toeplitz 矩阵 `W`，其每一行对应于卷积核在输入图像某位置的一组权重，无重叠区域填充为 0。
3. 互相关运算转化为一次稀疏矩阵与向量乘法：
   ```text
   y_vec = W · x_vec
   ```

---

### 练习 4：手工设计一些卷积核

#### 4.1 二阶导数的核的形式是什么？

* **一维情况推导**：
  一阶前向与后向差分分别为：
  ```text
  f'(x)   ≈ f(x+1) - f(x)
  f'(x-1) ≈ f(x) - f(x-1)
  ```
  二阶导数即一阶导数的差分：
  ```text
  f''(x) ≈ f'(x) - f'(x-1) 
         = (f(x+1) - f(x)) - (f(x) - f(x-1))
         = f(x+1) - 2·f(x) + f(x-1)
  ```
  对应的一维二阶导数核为：
  ```text
  K = [1, -2, 1]
  ```

* **二维图像情况（拉普拉斯算子 ∇²f = ∂²f/∂x² + ∂²f/∂y²）**：
  将水平二阶导数与垂直二阶导数相加，得到经典的二维拉普拉斯核（4-邻域）：
  ```text
  K_Laplacian = [[ 0,  1,  0],
                 [ 1, -4,  1],
                 [ 0,  1,  0]]
  ```
  若考虑对角线（8-邻域），形式为：
  ```text
  K_Laplacian8 = [[ 1,  1,  1],
                  [ 1, -8,  1],
                  [ 1,  1,  1]]
  ```

---

#### 4.2 积分的核的形式是什么？

* **原理**：
  离散信号中的“积分”对应于**累加求和**或**加权平滑**。
* **局部窗口积分（盒式滤波器 Box Filter / 均值滤波）**：
  如果计算大小为 `m × n` 局部邻域内的积分（总和），卷积核为全 1 矩阵：
  ```text
  K_int = torch.ones((m, n))
  ```
  若需计算归一化的积分（平均值），则乘以尺度系数：
  ```text
  K_mean = (1 / (m · n)) · torch.ones((m, n))
  ```
* **一维因果累加（前缀和）**：
  对于一维离散信号从起点到当前的累积积分，等价于与阶跃序列（全 1 核 `[1, 1, ..., 1]`）做卷积。

---

#### 4.3 得到 d 次导数的最小核的大小是多少？

* **一维最小核大小**：
  * **0 次导数**（恒等核）：大小为 `1`（核为 `[1]`）。
  * **1 次导数**（一阶差分）：大小为 `2`（如 `[1, -1]`）。
  * **2 次导数**（二阶差分）：大小为 `3`（如 `[1, -2, 1]`）。
  * **规律推导**：
    由于每一次离散求导本质上就是与一阶差分核 `[-1, 1]`（长度为 2）做一次卷积。
    两个长度分别为 `L_1` 和 `L_2` 的核卷积后，输出核长度为 `L_1 + L_2 - 1`。
    因此，连续进行 `d` 次一阶差分卷积，得到的最小核长度为：
    ```text
    最小核长度 = d · 2 - (d - 1) = d + 1
    ```
  * **结论**：得到 `d` 次导数的一维最小核尺寸是 **`d + 1`**。

* **二维各向同性对称核**：
  在二维图像处理中，若要求卷积核具有空间旋转对称性与几何中心点（奇数对称核），则：
  * 1 阶对称导数（如 Sobel）通常采用 `3 × 3`；
  * `d` 次导数的对称核最小尺寸通常要求为不小于 `d + 1` 的最小奇数，即为 `(d + 1) × (d + 1)`（当 `d` 为偶数时）。